# Encoder v3

Uses new encoding table, with active stimuli seperated as own type, emotional valences and male/female phenotypes on localizers.

This code takes as input the psychopy output runx.csv file and encodes stimuli codes in runx_stim_codes.txt file. The goal is to categories every stimulus and fixation according to the previously established encoding table (see excel table, stimuli = type + method + emotional valance). Also view excel for assumptions. The codes can then be passed into mne event object.

Decoder also included for some data counting an manual verification of attributed codes.

This script is coded to be executed for different runs with minor adjustements:  
- change input and output paths
- set localizer variable according to run

Possible improvements:
- add maps to encoder for clearer code

In [1]:
import numpy as np
import csv
import re

In [2]:
localizer = False

In [3]:
csv_path = r"C:\ .. \run3.csv"

Read csv and parse the column 'stim_file' (containting stimuli type information) into array

In [4]:
stim_files = []

with open(csv_path, mode ='r') as file:
    csvFile = csv.reader(file, delimiter=';')

    header = next(csvFile)
    stim_col = header.index("stim_file")

    for line in csvFile:
        stim_files.append(line[stim_col])

Encoding dictionnary containing type_code and emotion_code for each stimuli file name. See encoder excel.

In [5]:
encoder_dict = {
    'face': {
        'base': 100,
        'emotions': {
            '1' : 3, # anger
            '2' : 3,
            '3' : 3,
            '4' : 4, # fear
            '5' : 4,
            '6' : 4,
            '7' : 5, # happy
            '8' : 5,
            '9' : 5,
            '10' : 5,
            '11' : 2, # neutral
            '12' : 2,
            '13' : 2,
            '14' : 2,
            '15' : 2,
            '16' : 2,
            '17' : 2,
            '18' : 2,
            '19' : 2,
            '20' : 2,
            'AM' : 6, # male (for localizer)
            'AF' : 7, # female (for localizer)
        }
    },
    'body': {
        'base': 200,
        'emotions': {
            '1' : 4,
            '2' : 4,
            '3' : 4,
            '4' : 4,
            '5' : 3,
            '6' : 3,
            '7' : 3,
            '8' : 5,
            '9' : 5,
            '10' : 5,
            '11' : 2,
            '12' : 2,
            '13' : 2,
            '14' : 2,
            '15' : 2,
            '16' : 2,
            '17' : 2,
            '18' : 2,
            '19' : 2,
            '20' : 4,
            'male' : 6, # for localizer
            'female' : 7, # for localizer
        }
    },
    'object': {
        'base': 300,
        'emotions': {}
    },
    'house': {
        'base': 400,
        'emotions': {}
    },
}

For every stimulus, it iterates over the dictionary untill it finds the one corresponding to file_name first instance. Compared to not iterating over the dicitonnary, it's more work for the PC but cleaner code (less ifs).  

9 is default value if it can't find match, so if you see 9 somewhere in the encoded stimuli it messed up

re.search() checks for a match anywhere in the string  
https://docs.python.org/3/library/re.html#search-vs-match

The pattern r'\d+' means:  
r'' --> raw string, backslashes are treated literally  
\d --> matches any digit  
plus --> one or more of the preceding character

data['emotions'].get(gender.group(), 0) explained:  
.get(key, default) --> is a dictionary method that looks up key and returns its value if found, or the default value if not found  
.group() --> re.search() returns à match object <re.Match object; span=(0, 2), match='AM'>, so .group() is used to extract it and serve as key

In [26]:
codes = np.zeros(len(stim_files), dtype=np.int_)

for i, file_name in enumerate(stim_files):
    # Fixations
    if file_name == 'None':
        codes[i] = 600

    # Stimuli    
    else:
        split_fn = re.split(r'[_.]', file_name)

        # Actives
        if split_fn[0].endswith('B'):
            codes[i] = 500
            
        else:
            # Type and emotional valence for Faces, Bodies, Objects, Houses
            for category, data in encoder_dict.items():
                #print(category)
                
                if split_fn[0].startswith(category):
                    #print(data)
                    # Type
                    codes[i] += data['base']

                    # Emotional valence
                    if category in ('object','house') :
                        # Neutral (corrected to scrambled later if needed)
                        codes[i] += 2
                        break
                    elif localizer:
                        gender = re.search(r'AM|AF|female|male', split_fn[1])
                        if gender :
                            codes[i] += data['emotions'].get(gender.group(), 9)
                            # If gender is bc localizer contains scr stimuli like 'body08_scr_static.jpg' who don't have gender attributed but we don't care about their emov either
                        break
                    else :
                        number = re.search(r'\d+', split_fn[0])
                        if number:
                            codes[i] += data['emotions'].get(number.group(), 9)
                        else:
                            print(f'[{i}] Error: could not id emotion')
                        break

        # Corrects emotional valence if scrambled variant also assigns neutral emov for houses and objects ?????
        for part in split_fn:
            if part == 'scr':
                # remove possible previous emotional valence and replace by scrambled emotional valence (+1)
                codes[i] = codes[i] - codes[i]%10 + 1
                break

        #Method
        if localizer:
            codes[i] += 30
        elif split_fn[-1] in ('jpeg', 'jpg') :
            #Static
            codes[i] += 10
        elif split_fn[-1] == 'mp4':
            #Dynamic
            codes[i] += 20
        else:
            print(f'[{i}] Error: no method registered')

    print(f"[{i}] {file_name} --> {codes[i]}")

[0] None --> 600
[1] face17_static.jpeg --> 112
[2] None --> 600
[3] object12_static.jpeg --> 312
[4] None --> 600
[5] body16_scr_static.jpeg --> 211
[6] None --> 600
[7] face16_static.jpeg --> 112
[8] None --> 600
[9] object19_static.jpeg --> 312
[10] None --> 600
[11] body14B_scr_static.jpeg --> 511
[12] None --> 600
[13] body11_static.jpeg --> 212
[14] None --> 600
[15] body8_static.jpeg --> 215
[16] None --> 600
[17] body13_scr_static.jpeg --> 211
[18] None --> 600
[19] object17_scr_static.jpeg --> 311
[20] None --> 600
[21] body15_scr_static.jpeg --> 211
[22] None --> 600
[23] body4_static.jpeg --> 214
[24] None --> 600
[25] object11_scr_static.jpeg --> 311
[26] None --> 600
[27] object16_scr_static.jpeg --> 311
[28] None --> 600
[29] face20_scr_static.jpeg --> 111
[30] None --> 600
[31] face11_static.jpeg --> 112
[32] None --> 600
[33] object11_static.jpeg --> 312
[34] None --> 600
[35] object14_static.jpeg --> 312
[36] None --> 600
[37] body3_scr_static.jpeg --> 211
[38] None --

Exporting to txt file

In [28]:
import os

output_path = r"C:\ .. \stim_codes_run3_v3.txt"

with open(output_path, 'w') as f:
    for code in codes:
        f.write(f"{code}\n")

## Decoder
This section can answer some data related questions such as:  
-How many of each type [faces, objects, bodies and houses] ?  
-How many of each method [static, dynamic, localizer] ?  
Total number of stimuli and fixation can be used to check against number in prt file.

Flat dictionary of tuples chosen to loop over all entries cleanly

In [23]:
def init_counter_dict():

    counter_dict = {
        # Face
        ('face', 'static',    'scrambled'): 0,
        ('face', 'static',    'neutral'):   0,
        ('face', 'static',    'anger'):     0,
        ('face', 'static',    'fear'):      0,
        ('face', 'static',    'happy'):     0,
        ('face', 'static',    'male'):      0,
        ('face', 'static',    'female'):    0,
        ('face', 'dynamic',   'scrambled'): 0,
        ('face', 'dynamic',   'neutral'):   0,
        ('face', 'dynamic',   'anger'):     0,
        ('face', 'dynamic',   'fear'):      0,
        ('face', 'dynamic',   'happy'):     0,
        ('face', 'dynamic',   'male'):      0,
        ('face', 'dynamic',   'female'):    0,
        ('face', 'localizer', 'scrambled'): 0,
        ('face', 'localizer', 'neutral'):   0,
        ('face', 'localizer', 'anger'):     0,
        ('face', 'localizer', 'fear'):      0,
        ('face', 'localizer', 'happy'):     0,
        ('face', 'localizer', 'male'):      0,
        ('face', 'localizer', 'female'):    0,
        # Body
        ('body', 'static',    'scrambled'): 0,
        ('body', 'static',    'neutral'):   0,
        ('body', 'static',    'anger'):     0,
        ('body', 'static',    'fear'):      0,
        ('body', 'static',    'happy'):     0,
        ('body', 'static',    'male'):      0,
        ('body', 'static',    'female'):    0,
        ('body', 'dynamic',   'scrambled'): 0,
        ('body', 'dynamic',   'neutral'):   0,
        ('body', 'dynamic',   'anger'):     0,
        ('body', 'dynamic',   'fear'):      0,
        ('body', 'dynamic',   'happy'):     0,
        ('body', 'dynamic',   'male'):      0,
        ('body', 'dynamic',   'female'):    0,
        ('body', 'localizer', 'scrambled'): 0,
        ('body', 'localizer', 'neutral'):   0,
        ('body', 'localizer', 'anger'):     0,
        ('body', 'localizer', 'fear'):      0,
        ('body', 'localizer', 'happy'):     0,
        ('body', 'localizer', 'male'):      0,
        ('body', 'localizer', 'female'):    0,
        # Object
        ('object', 'static',    'scrambled'): 0,
        ('object', 'static',    'neutral'):   0,
        ('object', 'dynamic',   'scrambled'): 0,
        ('object', 'dynamic',   'neutral'):   0,
        ('object', 'localizer', 'scrambled'): 0,
        ('object', 'localizer', 'neutral'):   0,
        # House
        ('house', 'static',    'scrambled'): 0,
        ('house', 'static',    'neutral'):   0,
        ('house', 'dynamic',   'scrambled'): 0,
        ('house', 'dynamic',   'neutral'):   0,
        ('house', 'localizer', 'scrambled'): 0,
        ('house', 'localizer', 'neutral'):   0,
        # Active
        ('active',): 0,
        # Fixation
        ('fixation',): 0,
    }
    
    type_map   = {1: 'face', 2: 'body', 3: 'object', 4: 'house'}
    method_map = {1: 'static', 2: 'dynamic', 3: 'localizer'}
    emotion_map = {1: 'scrambled', 2: 'neutral', 3: 'anger', 4: 'fear', 5: 'happy', 6: 'male', 7:'female'}

    return counter_dict, type_map, method_map, emotion_map

In [27]:
counter_dict, type_map, method_map, emotion_map = init_counter_dict() #this ensures the counter is always reset to zero when exeuting decoder

for i, code in enumerate(codes):
    type_code    = int(code/100)
    method_code  = int((code%100)/10)
    emotion_code = int(code%10)

    if type_code == 6:
        counter_dict[('fixation',)] += 1

    elif type_code == 5:
        counter_dict[('active',)] += 1
        
    elif type_code in type_map and method_code in method_map and emotion_code in emotion_map:
        key = (type_map[type_code], method_map[method_code], emotion_map[emotion_code])
        if key in counter_dict:
            counter_dict[key] += 1
        else:
            print(f'[{i}] Error: key {key} not found in counter_dict')
    else:
        print(f'[{i}] Error: could not attribute code {code}')

# Types
total_faces    = sum(v for (type_, *_), v in counter_dict.items() if type_ == 'face')
total_bodies   = sum(v for (type_, *_), v in counter_dict.items() if type_ == 'body')
total_objects  = sum(v for (type_, *_), v in counter_dict.items() if type_ == 'object')
total_houses   = sum(v for (type_, *_), v in counter_dict.items() if type_ == 'house')
total_fixation = counter_dict[('fixation',)]
total_stimuli  = total_faces + total_bodies + total_objects + total_houses + counter_dict[('active',)] #Includes active stimuli to match the "stimuli" number from psychopy output

print(f"Types : Faces {total_faces} , Bodies {total_bodies} , Objects {total_objects} , Houses {total_houses} , Actives {counter_dict[('active',)]} , Stimuli {total_stimuli} , Fixations {total_fixation}")

# Methods
total_faces_static = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'face' and key[1] == 'static')
total_faces_dynamic = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'face' and key[1] == 'dynamic')
total_faces_localizer = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'face' and key[1] == 'localizer')
print(f"Faces Methods : Static {total_faces_static}, Dynamic {total_faces_dynamic}, Localizer {total_faces_localizer}")

total_bodies_static = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'body' and key[1] == 'static')
total_bodies_dynamic = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'body' and key[1] == 'dynamic')
total_bodies_localizer = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'body' and key[1] == 'localizer')
print(f"Bodies Methods : Static {total_bodies_static}, Dynamic {total_bodies_dynamic}, Localizer {total_bodies_localizer}")

total_objects_static = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'object' and key[1] == 'static')
total_objects_dynamic = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'object' and key[1] == 'dynamic')
total_objects_localizer = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'object' and key[1] == 'localizer')
print(f"Objects Methods : Static {total_objects_static}, Dynamic {total_objects_dynamic}, Localizer {total_objects_localizer}")

total_houses_static = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'house' and key[1] == 'static')
total_houses_dynamic = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'house' and key[1] == 'dynamic')
total_houses_localizer = sum(v for key, v in counter_dict.items() if len(key) >=2 and key[0] == 'house' and key[1] == 'localizer')
print(f"Houses Methods : Static {total_houses_static}, Dynamic {total_houses_dynamic}, Localizer {total_houses_localizer}")

# Full dictionary for methods and emotional valence
for key, count in counter_dict.items():
    print(f"{key}: {count}")

Types : Faces 47 , Bodies 49 , Objects 50 , Houses 0 , Actives 4 , Stimuli 150 , Fixations 151
Faces Methods : Static 47, Dynamic 0, Localizer 0
Bodies Methods : Static 49, Dynamic 0, Localizer 0
Objects Methods : Static 50, Dynamic 0, Localizer 0
Houses Methods : Static 0, Dynamic 0, Localizer 0
('face', 'static', 'scrambled'): 23
('face', 'static', 'neutral'): 14
('face', 'static', 'anger'): 3
('face', 'static', 'fear'): 3
('face', 'static', 'happy'): 4
('face', 'static', 'male'): 0
('face', 'static', 'female'): 0
('face', 'dynamic', 'scrambled'): 0
('face', 'dynamic', 'neutral'): 0
('face', 'dynamic', 'anger'): 0
('face', 'dynamic', 'fear'): 0
('face', 'dynamic', 'happy'): 0
('face', 'dynamic', 'male'): 0
('face', 'dynamic', 'female'): 0
('face', 'localizer', 'scrambled'): 0
('face', 'localizer', 'neutral'): 0
('face', 'localizer', 'anger'): 0
('face', 'localizer', 'fear'): 0
('face', 'localizer', 'happy'): 0
('face', 'localizer', 'male'): 0
('face', 'localizer', 'female'): 0
('body